In [1]:
import sys, types

# Pune un stub gol in sys.modules ca importul real al flash_attn sa nu mai fie incercat
# Dar trebuie sa lipseasca atributele asteptate -> mai bine il facem sa apara ca "nu exista"
# Trick: punem un finder care blocheaza importul
class _BlockFlashAttn:
    def find_spec(self, name, path=None, target=None):
        if name == "flash_attn" or name.startswith("flash_attn."):
            raise ImportError("flash_attn blocked manually")
        return None

sys.meta_path.insert(0, _BlockFlashAttn())

# Asigura-te ca nu e deja importat
for mod in list(sys.modules):
    if mod == "flash_attn" or mod.startswith("flash_attn"):
        del sys.modules[mod]

# Patch direct functia check inainte sa fie importata de modeling_whisper
import transformers.utils.import_utils as _iu
_iu.is_flash_attn_2_available = lambda: False
# Si in modulul care o re-exporta
import transformers.utils as _tu
if hasattr(_tu, "is_flash_attn_2_available"):
    _tu.is_flash_attn_2_available = lambda: False

print("flash_attn blocked. OK")

flash_attn blocked. OK


In [2]:
import os, gc, torch, pandas as pd, numpy as np
from dataclasses import dataclass
from typing import Any, Dict, List, Union

from datasets import Dataset, Audio
from transformers import (
    WhisperFeatureExtractor, WhisperTokenizer, WhisperProcessor,
    WhisperForConditionalGeneration, Seq2SeqTrainer, Seq2SeqTrainingArguments,
)
from peft import LoraConfig, get_peft_model, PeftModel
import evaluate

MODEL_NAME    = "openai/whisper-large-v3-turbo"
LANGUAGE      = "romanian"
TASK          = "transcribe"
CSV_PATH      = "./Dataset/transcriptions.csv"
AUDIO_ROOT    = "./Dataset"
OUTPUT_DIR    = "./whisper-turbo-ro-lora"
SAMPLING_RATE = 16000

print("torch:", torch.__version__, "| GPU:", torch.cuda.get_device_name(0))

torch: 2.6.0+cu124 | GPU: Tesla T4


/opt/tljh/user/lib/python3.10/site-packages/torch/cuda/__init__.py:734: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


In [4]:
df = pd.read_csv(CSV_PATH)
print("Total rows:", len(df))

df["audio"] = df["wav_path"].apply(lambda p: os.path.normpath(os.path.join(AUDIO_ROOT, str(p))))

exists_mask = df["audio"].apply(os.path.isfile)
print("Fisiere lipsa:", (~exists_mask).sum())
df = df[exists_mask].copy()

df = df[df["text"].notna()]
df["text"] = df["text"].astype(str).str.strip()
df = df[df["text"] != ""].copy()
print("Dupa filter text non-vid + fisiere existente:", len(df))

# Split
df = df.sample(frac=1.0, random_state=42).reset_index(drop=True)
n_eval = max(50, int(0.05 * len(df)))
eval_df  = df.iloc[:n_eval].reset_index(drop=True)
train_df = df.iloc[n_eval:].reset_index(drop=True)
print(f"Train: {len(train_df)} | Eval: {len(eval_df)}")

Total rows: 551
Fisiere lipsa: 0
Dupa filter text non-vid + fisiere existente: 551
Train: 501 | Eval: 50


In [5]:
def to_hf_dataset(pdf):
    ds = Dataset.from_pandas(pdf[["audio", "text"]], preserve_index=False)
    ds = ds.cast_column("audio", Audio(sampling_rate=SAMPLING_RATE))
    return ds

train_ds = to_hf_dataset(train_df)
eval_ds  = to_hf_dataset(eval_df)
print(train_ds)
print(eval_ds)

Dataset({
    features: ['audio', 'text'],
    num_rows: 501
})
Dataset({
    features: ['audio', 'text'],
    num_rows: 50
})


In [6]:
feature_extractor = WhisperFeatureExtractor.from_pretrained(MODEL_NAME)
tokenizer         = WhisperTokenizer.from_pretrained(MODEL_NAME, language=LANGUAGE, task=TASK)
processor         = WhisperProcessor.from_pretrained(MODEL_NAME, language=LANGUAGE, task=TASK)
print("Processor OK. Vocab size:", tokenizer.vocab_size)

Processor OK. Vocab size: 50257


In [7]:
MAX_INPUT_SECONDS = 30
MAX_LABEL_TOKENS  = 448

def prepare(batch):
    audio = batch["audio"]
    arr = audio["array"]
    sr  = audio["sampling_rate"]
    # safety: trunchiaza la 30s
    max_len = MAX_INPUT_SECONDS * sr
    if len(arr) > max_len:
        arr = arr[:max_len]
    feats = feature_extractor(arr, sampling_rate=sr).input_features[0]
    labels = tokenizer(batch["text"], truncation=True, max_length=MAX_LABEL_TOKENS).input_ids
    return {"input_features": feats, "labels": labels}

train_ds = train_ds.map(prepare, remove_columns=train_ds.column_names, num_proc=1, desc="prep train")
eval_ds  = eval_ds.map(prepare,  remove_columns=eval_ds.column_names,  num_proc=1, desc="prep eval")
print("Done.")

prep train:   0%|          | 0/501 [00:00<?, ? examples/s]

prep eval:   0%|          | 0/50 [00:00<?, ? examples/s]

Done.


In [8]:
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features):
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        # remove leading BOS daca tokenizer-ul l-a adaugat (Whisper il pune singur)
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

In [9]:
try:
    wer_metric = evaluate.load("wer")
    def compute_wer(pred_str, label_str):
        return 100 * wer_metric.compute(predictions=pred_str, references=label_str)
except Exception as e:
    print("evaluate.load fallback la jiwer:", e)
    import jiwer
    def compute_wer(pred_str, label_str):
        return 100 * jiwer.wer(label_str, pred_str)

def compute_metrics(pred):
    pred_ids  = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = tokenizer.pad_token_id
    pred_str  = tokenizer.batch_decode(pred_ids,  skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    return {"wer": compute_wer(pred_str, label_str)}

In [10]:
# IMPORTANT: attn_implementation="eager" ca sa nu mai incerce flash-attn nici la nivel de model
model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    attn_implementation="eager",
)
model.config.forced_decoder_ids  = None
model.config.suppress_tokens     = []
model.config.use_cache           = False

# Pentru fp16 + gradient checkpointing pe PEFT trebuie input grads
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

# Generation config pentru limba romana
model.generation_config.language = LANGUAGE
model.generation_config.task     = TASK
model.generation_config.forced_decoder_ids = processor.get_decoder_prompt_ids(
    language=LANGUAGE, task=TASK
)

lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 6,553,600 || all params: 815,431,680 || trainable%: 0.8037


In [12]:
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,        # effective batch = 16
    learning_rate=1e-4,
    warmup_ratio=0.05,
    num_train_epochs=3,
    fp16=True,
    bf16=False,
    gradient_checkpointing=True,
    eval_strategy="steps",                # in 4.46.3 e 'eval_strategy'
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    logging_steps=25,
    predict_with_generate=True,
    generation_max_length=225,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    remove_unused_columns=False,          # CRITIC pentru PEFT
    label_names=["labels"],               # CRITIC pentru PEFT + Seq2SeqTrainer
    dataloader_num_workers=2,
    optim="adamw_torch",
)

In [13]:
trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor.feature_extractor,
)

gc.collect()
torch.cuda.empty_cache()

trainer.train()

/tmp/ipykernel_672681/2049508528.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
/opt/tljh/user/lib/python3.10/site-packages/torch/utils/checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


Step,Training Loss,Validation Loss


TrainOutput(global_step=93, training_loss=0.6490602185649257, metrics={'train_runtime': 566.9062, 'train_samples_per_second': 2.651, 'train_steps_per_second': 0.164, 'total_flos': 2.55943536279552e+18, 'train_loss': 0.6490602185649257, 'epoch': 2.9682539682539684})

In [14]:
model.save_pretrained(OUTPUT_DIR)         # salveaza doar adapter-ul LoRA (~ MB)
processor.save_pretrained(OUTPUT_DIR)
print("Saved to", OUTPUT_DIR)

Saved to ./whisper-turbo-ro-lora


In [15]:
from transformers import pipeline

base = WhisperForConditionalGeneration.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16, attn_implementation="eager"
).to("cuda")
peft_model = PeftModel.from_pretrained(base, OUTPUT_DIR).to("cuda")
peft_model.eval()

pipe = pipeline(
    task="automatic-speech-recognition",
    model=peft_model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    chunk_length_s=30,
    device=0,
    torch_dtype=torch.float16,
)
sample_path = eval_df.iloc[0]["audio"]
ref = eval_df.iloc[0]["text"]
hyp = pipe(sample_path, generate_kwargs={"language": LANGUAGE, "task": TASK})["text"]
print("REF:", ref[:250])
print("HYP:", hyp[:250])

/home/jupyter-ivan/.local/lib/python3.10/site-packages/transformers/models/whisper/generation_whisper.py:509: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(
You have passed task=transcribe, but also have set `forced_decoder_ids` to [[1, None], [2, 50360]] which creates a conflict. `forced_decoder_ids` will be ignored in favor of task=transcribe.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


REF: Jurnalistii. Cum? Ca să nu se ibe tare, strânși de govern, de strângi și liberi. Trebuie să fie liber. Deci e adevărat.
HYP:  Jurnaliștii. Cum, în cazul și de tare, strânș de govern, de strâng, și liber. Trebuie să fie liber. Deci e adevărat.


In [16]:
from IPython.display import Audio as IPyAudio, display
import soundfile as sf

idx = 0  # schimba indexul ca sa asculti alt sample
sample_path = eval_df.iloc[idx]["audio"]
ref = eval_df.iloc[idx]["text"]

arr, sr = sf.read(sample_path)
print(f"File: {sample_path}")
print(f"Duration: {len(arr)/sr:.2f}s | SR: {sr}")
print("REF:", ref)

hyp = pipe(sample_path, generate_kwargs={"language": LANGUAGE, "task": TASK})["text"]
print("HYP:", hyp)

display(IPyAudio(sample_path))

File: Dataset/ep_8293_seg_019.wav
Duration: 9.54s | SR: 16000
REF: Jurnalistii. Cum? Ca să nu se ibe tare, strânși de govern, de strângi și liberi. Trebuie să fie liber. Deci e adevărat.


/home/jupyter-ivan/.local/lib/python3.10/site-packages/transformers/models/whisper/generation_whisper.py:509: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(


HYP:  Jurnaliștii. Cum, în cazul și de tare, strânș de govern, de strâng, și liber. Trebuie să fie liber. Deci e adevărat.


In [30]:
from IPython.display import HTML, Javascript, display
import base64, os

RECORDING_PATH = "./records/mic_recording.wav"

HTML_REC = """
<div style="font-family:sans-serif">
  <button id="recBtn" style="padding:10px 20px;font-size:16px;background:#e74c3c;color:white;border:none;border-radius:6px;cursor:pointer">
    🎤 Start Recording
  </button>
  <span id="recStatus" style="margin-left:15px;color:#555"></span>
  <br><br>
  <audio id="recPlayer" controls style="display:none"></audio>
  <br>
  <textarea id="recB64" style="display:none"></textarea>
</div>

<script>
(function(){
  let mediaRecorder, chunks = [], stream;
  const btn = document.getElementById('recBtn');
  const status = document.getElementById('recStatus');
  const player = document.getElementById('recPlayer');
  const b64ta  = document.getElementById('recB64');

  btn.onclick = async () => {
    if (!mediaRecorder || mediaRecorder.state === 'inactive') {
      stream = await navigator.mediaDevices.getUserMedia({audio: true});
      chunks = [];
      mediaRecorder = new MediaRecorder(stream);
      mediaRecorder.ondataavailable = e => chunks.push(e.data);
      mediaRecorder.onstop = async () => {
        const blob = new Blob(chunks, {type: 'audio/webm'});
        const url  = URL.createObjectURL(blob);
        player.src = url; player.style.display = 'block';
        const buf = await blob.arrayBuffer();
        const b64 = btoa(String.fromCharCode(...new Uint8Array(buf)));
        b64ta.value = b64;
        status.textContent = '✅ Recorded ' + (buf.byteLength/1024).toFixed(1) + ' KB. Ruleaza celula urmatoare.';
        stream.getTracks().forEach(t => t.stop());
      };
      mediaRecorder.start();
      btn.textContent = '⏹ Stop';
      status.textContent = '🔴 Recording...';
    } else {
      mediaRecorder.stop();
      btn.textContent = '🎤 Start Recording';
    }
  };
})();
</script>
"""
display(HTML(HTML_REC))

In [31]:
import subprocess, base64

# Converteste la wav 16kHz mono (Whisper vrea 16k)
subprocess.run([
    "ffmpeg", "-y", "-i", "./records/mic_recording.webm",
    "-ar", "16000", "-ac", "1", RECORDING_PATH
], check=True, capture_output=True)

# Asculta
from IPython.display import Audio as IPyAudio, display
display(IPyAudio(RECORDING_PATH))

# Transcribe cu modelul fine-tuned
result = pipe(RECORDING_PATH, generate_kwargs={"language": LANGUAGE, "task": TASK})
print("📝 Transcript:", result["text"])

/home/jupyter-ivan/.local/lib/python3.10/site-packages/transformers/models/whisper/generation_whisper.py:509: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(


📝 Transcript:  Jó, se va, nu, să lega is.
